In [12]:
# EDA
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, f_oneway, ttest_ind
from colorama import Fore, Back, Style

# Visualização
import plotly.express as px
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

# ML
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report, confusion_matrix, \
                            ConfusionMatrixDisplay, log_loss, roc_curve, roc_auc_score

# Interpretabilidade
import shap

# Carregar os dados

In [13]:
df_empresas = pd.read_csv('datasets/companies_profile.csv')

# Análise Inicial

In [14]:
# Visualizar a estrutura
df_empresas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 22 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   ID                         1000 non-null   int64  
 1   Nome_Empresa               1000 non-null   object 
 2   Receita_Anual              1000 non-null   int64  
 3   Margem_Liquida             1000 non-null   float64
 4   Endividamento              1000 non-null   float64
 5   Setor                      1000 non-null   object 
 6   Regiao                     1000 non-null   object 
 7   Tempo_Operacao             1000 non-null   int64  
 8   Auditoria_Externa          1000 non-null   int64  
 9   Rating_Credito             1000 non-null   float64
 10  Tipo_Empresa               1000 non-null   object 
 11  Politica_Sustentabilidade  1000 non-null   object 
 12  Estrategia_Expansao        1000 non-null   object 
 13  Gestao_Risco               1000 non-null   object

In [15]:
# Visualizar primeiras linhas
df_empresas.head(10)

,ID,Nome_Empresa,Receita_Anual,Margem_Liquida,Endividamento,Setor,Regiao,Tempo_Operacao,Auditoria_Externa,Rating_Credito,...,Estrategia_Expansao,Gestao_Risco,Cobertura_Seguros,Maturidade_Digital,Governanca_Corporativa,Cultura_Inovacao,Relacao_Comunidade,Risco_Credito,Risco_Compliance,Risco_Mercado
0,1,Hahn Group,6523388,0.482879,1.032767,Manufatura,Europa,26,0,0.938715,...,Parcerias,Centralizada,Básica,Avançada,Fraca,Neutra,Regular,0,0,0
1,2,Lopez Group,6650634,0.121292,0.492841,Tecnologia,Europa,20,1,0.492362,...,Orgânica,Centralizada,Básica,Inicial,Média,Neutra,Ruim,0,0,0
2,3,Sparks and Sons,4404572,0.190114,0.757099,Manufatura,América Latina,6,0,0.700866,...,Orgânica,Decentralizada,Básica,Inicial,Alta,Neutra,Boa,0,0,0
3,4,"Fields, Ramirez and Craig",2334489,0.402442,2.327962,Manufatura,Europa,6,1,0.855551,...,Parcerias,Centralizada,Nenhuma,Intermediária,Média,Inovadora,Excelente,1,0,0
4,5,"Campbell, Hernandez and Lyons",9624682,0.174549,1.722357,Saúde,América do Norte,18,1,0.418291,...,Aquisições,Decentralizada,Nenhuma,Avançada,Fraca,Neutra,Regular,0,0,0
5,6,Robinson Ltd,7304212,-0.070876,1.290515,Saúde,América do Norte,4,1,0.309413,...,Orgânica,Centralizada,Ampla,Avançada,Alta,Neutra,Ruim,1,0,1
6,7,Bennett LLC,9728519,0.009717,0.130516,Tecnologia,Ásia,17,0,0.936725,...,Parcerias,Decentralizada,Básica,Avançada,Média,Neutra,Regular,0,1,0
7,8,"Rios, Stevens and Johnson",4572471,0.016956,2.481797,Saúde,Europa,14,1,0.515677,...,Parcerias,Centralizada,Nenhuma,Intermediária,Alta,Neutra,Regular,1,0,0
8,9,"Murphy, Walters and Cruz",4623669,0.078123,1.711492,Tecnologia,América Latina,14,0,0.497254,...,Aquisições,Decentralizada,Básica,Avançada,Média,Neutra,Ruim,0,1,0
9,10,Larson Ltd,7504852,0.098752,0.860614,Financeiro,América Latina,13,0,0.178416,...,Aquisições,Centralizada,Básica,Intermediária,Média,Conservadora,Boa,1,1,0


In [16]:
# Visualizar ultimas linhas
df_empresas.tail(10)

,ID,Nome_Empresa,Receita_Anual,Margem_Liquida,Endividamento,Setor,Regiao,Tempo_Operacao,Auditoria_Externa,Rating_Credito,...,Estrategia_Expansao,Gestao_Risco,Cobertura_Seguros,Maturidade_Digital,Governanca_Corporativa,Cultura_Inovacao,Relacao_Comunidade,Risco_Credito,Risco_Compliance,Risco_Mercado
990,991,Meza-Garcia,2701623,0.444208,2.263319,Manufatura,América do Norte,21,0,0.567443,...,Parcerias,Decentralizada,Nenhuma,Inicial,Média,Inovadora,Boa,1,0,0
991,992,Jacobs LLC,9866854,0.164538,0.469137,Manufatura,Europa,39,0,0.377008,...,Orgânica,Decentralizada,Básica,Avançada,Média,Inovadora,Regular,1,0,0
992,993,Smith LLC,6331219,0.231550,2.476207,Tecnologia,Europa,23,0,0.964929,...,Orgânica,Centralizada,Básica,Inicial,Média,Inovadora,Regular,1,0,0
993,994,Erickson Group,4208997,-0.090353,2.391633,Financeiro,América do Norte,27,1,0.273542,...,Aquisições,Centralizada,Nenhuma,Inicial,Fraca,Inovadora,Excelente,1,0,0
994,995,"Wilson, Nicholson and Benson",3464695,-0.169385,1.287220,Saúde,Ásia,45,1,0.399658,...,Parcerias,Centralizada,Nenhuma,Inicial,Média,Neutra,Regular,1,0,0
995,996,Kelley-Ali,1145482,-0.122434,1.330529,Financeiro,América do Norte,44,1,0.982860,...,Parcerias,Decentralizada,Nenhuma,Avançada,Média,Inovadora,Regular,0,0,0
996,997,Sawyer-Phillips,5886265,0.218972,1.255718,Saúde,Europa,18,1,0.536470,...,Orgânica,Decentralizada,Básica,Inicial,Alta,Inovadora,Regular,0,0,0
997,998,"Chang, Dudley and Lee",6008635,-0.133492,1.625065,Tecnologia,Europa,24,1,0.502630,...,Parcerias,Decentralizada,Nenhuma,Intermediária,Alta,Inovadora,Ruim,0,0,0
998,999,Bradshaw Inc,7587345,0.063195,0.317732,Tecnologia,Ásia,47,1,0.257778,...,Aquisições,Centralizada,Ampla,Intermediária,Alta,Neutra,Regular,1,0,0
999,1000,Schneider-Nguyen,9783910,0.185064,0.178283,Saúde,América do Norte,40,0,0.701364,...,Aquisições,Centralizada,Nenhuma,Avançada,Fraca,Inovadora,Ruim,0,0,0


In [17]:
# Valores possíveis para variáveis categoricas
for col in df_empresas.drop(columns=['Nome_Empresa']).select_dtypes(include=['object']).columns:
    print(f'\nValores únicos em {col}:')
    print(df_empresas[col].unique())


Valores únicos em Setor:
['Manufatura' 'Tecnologia' 'Saúde' 'Financeiro']

Valores únicos em Regiao:
['Europa' 'América Latina' 'América do Norte' 'Ásia']

Valores únicos em Tipo_Empresa:
['MEI' 'S.A.' 'Limitada' 'Multinacional']

Valores únicos em Politica_Sustentabilidade:
['Baixa' 'Alta' 'Média']

Valores únicos em Estrategia_Expansao:
['Parcerias' 'Orgânica' 'Aquisições']

Valores únicos em Gestao_Risco:
['Centralizada' 'Decentralizada']

Valores únicos em Cobertura_Seguros:
['Básica' 'Nenhuma' 'Ampla']

Valores únicos em Maturidade_Digital:
['Avançada' 'Inicial' 'Intermediária']

Valores únicos em Governanca_Corporativa:
['Fraca' 'Média' 'Alta']

Valores únicos em Cultura_Inovacao:
['Neutra' 'Inovadora' 'Conservadora']

Valores únicos em Relacao_Comunidade:
['Regular' 'Ruim' 'Boa' 'Excelente']


In [18]:
# Estatísticas das variáveis numéricas
df_empresas.describe()

,ID,Receita_Anual,Margem_Liquida,Endividamento,Tempo_Operacao,Auditoria_Externa,Rating_Credito,Risco_Credito,Risco_Compliance,Risco_Mercado
count,1000.000000,1.000000e+03,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,500.500000,4.992928e+06,0.152379,1.317909,25.367000,0.496000,0.487000,0.595000,0.206000,0.110000
std,288.819436,2.804931e+06,0.199511,0.700616,14.103873,0.500234,0.292846,0.491138,0.404633,0.313046
min,1.000000,1.393530e+05,-0.199834,0.100452,1.000000,0.000000,0.000748,0.000000,0.000000,0.000000
25%,250.750000,2.646178e+06,-0.021089,0.691918,13.000000,0.000000,0.232142,0.000000,0.000000,0.000000
50%,500.500000,5.032603e+06,0.160441,1.364206,25.000000,0.000000,0.475893,1.000000,0.000000,0.000000
75%,750.250000,7.270658e+06,0.323736,1.919788,38.000000,1.000000,0.742092,1.000000,0.000000,0.000000
max,1000.000000,9.989550e+06,0.499547,2.499313,49.000000,1.000000,0.999049,1.000000,1.000000,1.000000


# EDA

In [19]:
# Lista de features numéricas
features_numericas = df_empresas.drop(columns=['ID', 'Risco_Credito', 'Risco_Compliance', 'Risco_Mercado'], axis=1).select_dtypes(include=['float64', 'int64']).columns
features_numericas

Index(['Receita_Anual', 'Margem_Liquida', 'Endividamento', 'Tempo_Operacao',
       'Auditoria_Externa', 'Rating_Credito'],
      dtype='object')

In [20]:
# Visualizar a distribuição das features numéricas
for col in features_numericas:
    fig = px.histogram(df_empresas, x=col, nbins=20, title=f'Distribuição de {col}')
    fig.show()

In [21]:
# Lista de Features categóricas
features_categoricas = df_empresas.drop(columns=['Nome_Empresa']).select_dtypes(include=['object']).columns
features_categoricas

Index(['Setor', 'Regiao', 'Tipo_Empresa', 'Politica_Sustentabilidade',
       'Estrategia_Expansao', 'Gestao_Risco', 'Cobertura_Seguros',
       'Maturidade_Digital', 'Governanca_Corporativa', 'Cultura_Inovacao',
       'Relacao_Comunidade'],
      dtype='object')

In [22]:
# Contagem de valores para variáveis categóricas
for col in features_categoricas:
    df_count = df_empresas[col].value_counts().reset_index()
    df_count.columns = ['categoria', 'contagem']
    fig = px.bar(df_count,
                 x='categoria',
                 y='contagem',
                 title=f'Distribuição / Contagem de {col}')
    fig.show()

In [23]:
# Lista de Targets
targets = ['Risco_Credito', 'Risco_Compliance', 'Risco_Mercado']

In [24]:
# Visualizar Distribuição dos targets
for col in targets:
    df_count = df_empresas[col].value_counts().reset_index()
    df_count.columns = ['categoria', 'contagem']
    fig = px.bar(df_count,
                 x='categoria',
                 y='contagem',
                 title=f'Distribuição / Contagem de {col}')
    fig.show()

In [25]:
# Analisar relação entre features numericas e categoricas e os targets
for target in targets:
    for col in features_numericas:
        fig = px.box(df_empresas, x=target, y=col, title=f'{col} por {target}')
        fig.show()

    for col in features_categoricas:
        fig = px.histogram(df_empresas, x=col, color=target, barmode='group', title=f'{col} por {target}')
        fig.show()

In [28]:
# Matriz de Correlação
correlation_matrix = df_empresas[features_numericas.tolist() + targets].corr()

# Heatmap da matriz de correlação
fig = px.imshow(correlation_matrix,
                color_continuous_scale='armyrose',
                title='Matriz de Correlação',
                zmin=-1,
                zmax=1
)

# Ajustes no Heatmap
fig.update_traces(text=correlation_matrix, texttemplate='%{text:.1%}', textfont=dict(size=9))

fig.update_layout(
    width=1500,
    height=800,
    title_font=dict(size=20),
    font=dict(size=14)
)

fig.show()